[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/huggingface_ecosystem.ipynb)

# 01 . The Hugging Face Ecosystem from Scratch

## Concept — Hugging Face ecosystem (revision notes)

**What it is**
- The Hugging Face stack is four cooperating libraries: `tokenizers` (text -> ids), `datasets` (columnar, memory-mapped data with `.map`), `transformers` (model architectures + configs + `from_pretrained`) and `accelerate` (device / precision / distributed plumbing).
- Each one is a thin, well-defined contract. Once you can write a toy version of each contract you can read any HF code base.

**Why it matters**
- Nearly every open LLM finetuning recipe (TRL, PEFT, Axolotl) is written against these APIs. Knowing what a tokenizer, an attention mask, a `labels` tensor or `Accelerator.prepare` really does removes most of the "magic" and most of the bugs.

**When to use**
- Use the real libraries for production work. Re-implement the pieces from scratch (as here) to understand them, to debug mismatches, or when you need a tiny dependency-free pipeline.

**Math & intuition**
- Tokenization: BPE starts from characters/bytes and repeatedly merges the most frequent adjacent pair, so frequent words become one token and rare words fall back to pieces. Vocabulary size is the number of merges + base symbols.
- Batching: sequences are padded to equal length; an `attention_mask` (1 = real, 0 = pad) tells the model which positions to ignore and `labels = -100` tells the loss to ignore them.
- Causal LM loss: `logits[:, :-1]` predict `ids[:, 1:]` (labels are shifted by one).
- Config + weights: a model is `config.json` (architecture hyper-parameters) + a `state_dict` (tensors). `from_pretrained` = build from config, then load weights.

### 1. A character-level tokenizer from scratch

**What this cell does (plain language):** we build the simplest tokenizer: a vocabulary of every character seen in a corpus plus special tokens (`<pad>`, `<unk>`, `<bos>`, `<eos>`), with `encode` and `decode`. This is the same contract as `tokenizer.encode` / `tokenizer.decode` in `transformers`.

In [1]:
import torch, json, os, tempfile, random
torch.manual_seed(0); random.seed(0)

corpus = ["the cat sat on the mat", "the dog sat on the log", "a cat and a dog sat", "low lower lowest", "newer newest wider"]

class CharTokenizer:
    specials = ["<pad>", "<unk>", "<bos>", "<eos>"]
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.itos = self.specials + chars
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
    def __len__(self): return len(self.itos)
    def encode(self, text, add_special=True):
        ids = [self.stoi.get(ch, self.unk_id) for ch in text]   # unseen char -> <unk>
        return [self.bos_id] + ids + [self.eos_id] if add_special else ids
    def decode(self, ids, skip_special=True):
        return "".join(self.itos[i] for i in ids if not (skip_special and i < len(self.specials)))

tok = CharTokenizer(corpus)
ids = tok.encode("the cat")
print("vocab size :", len(tok))
print("ids        :", ids)
print("decoded    :", repr(tok.decode(ids)))
print("unseen char:", tok.encode("zebra", add_special=False), "-> 'z' maps to <unk> id", tok.unk_id)

vocab size : 20
ids        : [2, 18, 10, 8, 4, 6, 5, 18, 3]
decoded    : 'the cat'
unseen char: [1, 8, 1, 16, 5] -> 'z' maps to <unk> id 1


### 2. Byte-pair encoding (BPE) trained from scratch

**What this cell does (plain language):** we implement the real BPE training algorithm in about 20 lines: start from characters (with an end-of-word marker), count adjacent pairs across the corpus, merge the most frequent pair into a new symbol, and repeat. Watch frequent chunks like `lo`, `we`, `est` become single tokens.

In [2]:
from collections import Counter

def word_freqs(texts):
    c = Counter()
    for t in texts:
        for w in t.split():
            c[tuple(w) + ("</w>",)] += 1       # word as tuple of symbols + end marker
    return c

def get_pairs(vocab):
    pairs = Counter()
    for word, f in vocab.items():
        for a, b in zip(word, word[1:]):
            pairs[(a, b)] += f
    return pairs

def merge_pair(pair, vocab):
    out = {}
    for word, f in vocab.items():
        new, i = [], 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new.append(word[i] + word[i + 1]); i += 2   # fuse the pair
            else:
                new.append(word[i]); i += 1
        out[tuple(new)] = f
    return out

def train_bpe(texts, num_merges):
    vocab = word_freqs(texts)
    merges = []
    for _ in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs: break
        best = max(pairs, key=lambda p: (pairs[p], p))   # tie-break deterministically
        vocab = merge_pair(best, vocab)
        merges.append(best)
    return merges

def bpe_encode_word(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:                                   # apply merges in learned order
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == a and symbols[i + 1] == b:
                symbols[i:i + 2] = [a + b]
            else:
                i += 1
    return symbols

merges = train_bpe(corpus * 3, num_merges=15)
print("first merges:", merges[:8])
for w in ["lowest", "newest", "cat", "lowering"]:
    print(f"{w:9s} ->", bpe_encode_word(w, merges))

first merges: [('t', '</w>'), ('a', 't</w>'), ('w', 'e'), ('t', 'h'), ('th', 'e'), ('the', '</w>'), ('l', 'o'), ('s', 'at</w>')]
lowest    -> ['lo', 'west</w>']
newest    -> ['n', 'e', 'west</w>']
cat       -> ['c', 'at</w>']
lowering  -> ['lo', 'we', 'r', 'i', 'n', 'g</w>']


### 3. Wrapping BPE into a tokenizer with ids, special tokens and saving

**What this cell does (plain language):** we turn the merge list into a usable tokenizer (symbol -> id table), then save/load it as JSON, which is exactly what `tokenizer.json` / `save_pretrained` do. Round-tripping `decode(encode(x)) == x` is the first test any tokenizer should pass.

In [3]:
class BPETokenizer:
    specials = ["<pad>", "<unk>", "<bos>", "<eos>"]
    def __init__(self, merges, base_chars):
        self.merges = [tuple(m) for m in merges]
        symbols = list(base_chars) + ["</w>"] + ["".join(m) for m in self.merges]
        self.itos = self.specials + symbols
        self.stoi = {s: i for i, s in enumerate(self.itos)}
    def __len__(self): return len(self.itos)
    def encode(self, text, add_special=True):
        ids = []
        for w in text.split():
            ids += [self.stoi.get(s, 1) for s in bpe_encode_word(w, self.merges)]
        return [2] + ids + [3] if add_special else ids
    def decode(self, ids):
        s = "".join(self.itos[i] for i in ids if i >= 4)
        return s.replace("</w>", " ").strip()
    def save(self, path):
        json.dump({"merges": self.merges, "itos": self.itos}, open(path, "w"))
    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        t = cls.__new__(cls); t.merges = [tuple(m) for m in d["merges"]]
        t.itos = d["itos"]; t.stoi = {s: i for i, s in enumerate(t.itos)}
        return t

base = sorted(set("".join(corpus)) - {" "})
bpe = BPETokenizer(merges, base)
text = "the lowest cat"
ids = bpe.encode(text)
print("vocab size:", len(bpe), "| ids:", ids)
print("round trip:", repr(bpe.decode(ids)), "==", repr(text), bpe.decode(ids) == text)
with tempfile.TemporaryDirectory() as d:
    bpe.save(os.path.join(d, "tokenizer.json")); bpe2 = BPETokenizer.load(os.path.join(d, "tokenizer.json"))
print("reloaded gives same ids:", bpe2.encode(text) == ids)

vocab size: 35 | ids: [2, 25, 26, 31, 5, 21, 3]
round trip: 'the lowest cat' == 'the lowest cat' True
reloaded gives same ids: True


### 4. Batching: padding, attention mask, truncation and labels

**What this cell does (plain language):** we write the collate function that every HF `DataCollator` implements: pad a list of variable-length id lists to the batch max, build the `attention_mask`, and build `labels` where padded positions are `-100` so the loss skips them. We then verify the cross-entropy really ignores padding.

In [4]:
import torch.nn.functional as F

def collate(batch_ids, pad_id=0, max_len=None):
    if max_len: batch_ids = [x[:max_len] for x in batch_ids]       # truncation
    L = max(len(x) for x in batch_ids)
    input_ids = torch.full((len(batch_ids), L), pad_id)
    attention_mask = torch.zeros(len(batch_ids), L, dtype=torch.long)
    labels = torch.full((len(batch_ids), L), -100)                 # -100 = ignored by the loss
    for i, x in enumerate(batch_ids):
        input_ids[i, :len(x)] = torch.tensor(x)
        attention_mask[i, :len(x)] = 1
        labels[i, :len(x)] = torch.tensor(x)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

batch = collate([tok.encode("the cat"), tok.encode("a cat and a dog sat")])
for k, v in batch.items(): print(k, v.shape); print(v)

# Proof that -100 labels are skipped: loss over the padded batch == loss over only real tokens.
V = len(tok); logits = torch.randn(2, batch["input_ids"].shape[1], V)
lab = batch["labels"][:, 1:]; lg = logits[:, :-1]                   # shift: position t predicts token t+1
full = F.cross_entropy(lg.reshape(-1, V), lab.reshape(-1), ignore_index=-100)
keep = lab != -100
manual = F.cross_entropy(lg[keep], lab[keep])
print("loss with ignore_index:", full.item(), "| manual real-token loss:", manual.item())

input_ids torch.Size([2, 21])
tensor([[ 2, 18, 10,  8,  4,  6,  5, 18,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0],
        [ 2,  5,  4,  6,  5, 18,  4,  5, 14,  7,  4,  5,  4,  7, 15,  9,  4, 17,
          5, 18,  3]])
attention_mask torch.Size([2, 21])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
labels torch.Size([2, 21])
tensor([[   2,   18,   10,    8,    4,    6,    5,   18,    3, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100],
        [   2,    5,    4,    6,    5,   18,    4,    5,   14,    7,    4,    5,
            4,    7,   15,    9,    4,   17,    5,   18,    3]])
loss with ignore_index: 3.4133877754211426 | manual real-token loss: 3.4133877754211426


### 5. A tiny `datasets`-style Dataset with map / filter / split

**What this cell does (plain language):** we implement the core of `datasets.Dataset`: a columnar table (dict of lists) with `.map` (add or change columns), `.filter`, `.train_test_split` and `.with_format('torch')`-like indexing. Real `datasets` stores Arrow files on disk and caches `.map` results; the API shape is identical.

In [5]:
import random
class MiniDataset:
    def __init__(self, cols): self.cols = {k: list(v) for k, v in cols.items()}
    def __len__(self): return len(next(iter(self.cols.values())))
    def __getitem__(self, i):
        if isinstance(i, str): return self.cols[i]                 # ds["text"] -> column
        return {k: v[i] for k, v in self.cols.items()}              # ds[3] -> row dict
    @property
    def column_names(self): return list(self.cols)
    def map(self, fn, batched=False):
        if not batched:
            rows = [{**self[i], **fn(self[i])} for i in range(len(self))]
            return MiniDataset({k: [r[k] for r in rows] for k in rows[0]})
        out = fn(self.cols); return MiniDataset({**self.cols, **out})
    def filter(self, pred):
        keep = [i for i in range(len(self)) if pred(self[i])]
        return MiniDataset({k: [v[i] for i in keep] for k, v in self.cols.items()})
    def train_test_split(self, test_size=0.2, seed=0):
        idx = list(range(len(self))); random.Random(seed).shuffle(idx)
        n_test = int(len(idx) * test_size)
        pick = lambda ii: MiniDataset({k: [v[i] for i in ii] for k, v in self.cols.items()})
        return {"train": pick(idx[n_test:]), "test": pick(idx[:n_test])}

ds = MiniDataset({"text": corpus * 4, "label": [0, 1, 0, 1, 0] * 4})
ds = ds.map(lambda r: {"input_ids": bpe.encode(r["text"]), "n_tokens": len(bpe.encode(r["text"]))})
print("columns:", ds.column_names, "| rows:", len(ds))
print("row 0  :", ds[0])
ds_long = ds.filter(lambda r: r["n_tokens"] > 12)
print("rows with > 12 tokens:", len(ds_long))
split = ds.train_test_split(0.25)
print("train/test sizes:", len(split["train"]), len(split["test"]))

columns: ['text', 'label', 'input_ids', 'n_tokens'] | rows: 20
row 0  : {'text': 'the cat sat on the mat', 'label': 0, 'input_ids': [2, 25, 5, 21, 27, 34, 25, 12, 21, 3], 'n_tokens': 10}
rows with > 12 tokens: 8
train/test sizes: 15 5


### 6. Model config + save_pretrained / from_pretrained by hand

**What this cell does (plain language):** we embed the tiny GPT we will reuse throughout this folder, then reproduce the `save_pretrained` / `from_pretrained` contract: write `config.json` (architecture) and `model.pt` (weights), and rebuild an identical model from them. This is all the HF directory layout really is.

In [6]:
import math, torch, torch.utils.checkpoint, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass, asdict

@dataclass
class GPTConfig:
    vocab_size: int = 64
    block_size: int = 32
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64

class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.h = c.n_head
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # causal mask built in
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        self.attn = Attn(c)
        self.mlp = nn.Sequential(nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(), nn.Linear(4 * c.n_embd, c.n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos = nn.Embedding(c.block_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.use_ckpt = False
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)       # small init => starting loss ~ ln(vocab)
            if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None, ignore_index=-100):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for b in self.blocks:
            if self.use_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(b, x, use_reentrant=False)
            else:
                x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=ignore_index)
        return logits, loss

torch.manual_seed(0)
cfg = GPTConfig(vocab_size=len(bpe), block_size=32, n_layer=2, n_head=2, n_embd=32)
model = GPT(cfg)
print("parameters:", sum(p.numel() for p in model.parameters()))

def save_pretrained(model, d):
    os.makedirs(d, exist_ok=True)
    json.dump(asdict(model.cfg), open(os.path.join(d, "config.json"), "w"), indent=1)
    torch.save(model.state_dict(), os.path.join(d, "model.pt"))
def from_pretrained(d):
    cfg = GPTConfig(**json.load(open(os.path.join(d, "config.json"))))
    m = GPT(cfg); m.load_state_dict(torch.load(os.path.join(d, "model.pt"))); return m

x = torch.tensor([bpe.encode("the cat")])
with tempfile.TemporaryDirectory() as d:
    save_pretrained(model, d); print("files:", sorted(os.listdir(d)))
    m2 = from_pretrained(d)
model.eval(); m2.eval()
print("identical outputs after reload:", torch.allclose(model(x)[0], m2(x)[0]))

parameters: 27616
files: ['config.json', 'model.pt']
identical outputs after reload: True


### 7. Generation: greedy, temperature and top-k sampling

**What this cell does (plain language):** we write the `generate()` loop that sits behind `model.generate`: feed the tokens so far, take the last position's logits, pick the next token (argmax, or sample after temperature scaling and top-k filtering), append, repeat. The model is untrained so the text is gibberish, but the mechanics are the real ones.

In [7]:
@torch.no_grad()
def generate(model, idx, max_new=10, temperature=1.0, top_k=None, greedy=False):
    model.eval()
    for _ in range(max_new):
        ctx = idx[:, -model.cfg.block_size:]                 # crop to context window
        logits = model(ctx)[0][:, -1, :]                     # only the last position matters
        if greedy:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature                    # <1 sharper, >1 flatter
            if top_k:
                kth = logits.topk(top_k).values[:, -1, None]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            nxt = torch.multinomial(logits.softmax(-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

prompt = torch.tensor([bpe.encode("the cat", add_special=True)[:-1]])   # drop <eos> so the model continues
torch.manual_seed(1)
print("greedy      :", repr(bpe.decode(generate(model, prompt, greedy=True)[0].tolist())))
print("temp 1.0    :", repr(bpe.decode(generate(model, prompt)[0].tolist())))
print("temp 0.3,k=5:", repr(bpe.decode(generate(model, prompt, temperature=0.3, top_k=5)[0].tolist())))

greedy      : 'the cat'


temp 1.0    : 'the cat theigg oolomr'


temp 0.3,k=5: 'the cat    thether th'


### 8. A hand-written mini `Accelerator` (what accelerate does)

**What this cell does (plain language):** we implement the three jobs `accelerate.Accelerator` does for you: pick the device, wrap autocast (mixed precision) around the forward pass, and handle gradient accumulation by scaling the loss and stepping only every N micro-batches. We then use it in a short training loop on the tokenized dataset.

In [8]:
import contextlib
class MiniAccelerator:
    def __init__(self, grad_accum=1, mixed_precision=None):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.grad_accum, self.step_count = grad_accum, 0
        self.amp_dtype = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(mixed_precision)
    def prepare(self, model, opt): return model.to(self.device), opt
    def autocast(self):
        if self.amp_dtype is None: return contextlib.nullcontext()
        return torch.autocast(self.device.type, dtype=self.amp_dtype)
    @property
    def sync_gradients(self): return self.step_count % self.grad_accum == 0
    def backward(self, loss):
        self.step_count += 1
        (loss / self.grad_accum).backward()               # average over the accumulation window

def lm_batch(rows, block=16):
    b = collate([r["input_ids"] for r in rows], max_len=block)
    return b["input_ids"][:, :-1], b["labels"][:, 1:]

acc = MiniAccelerator(grad_accum=2, mixed_precision="bf16")
torch.manual_seed(0)
model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
model, opt = acc.prepare(model, opt)
train = split["train"]
losses = []
for step in range(60):
    rows = [train[(step * 4 + j) % len(train)] for j in range(4)]
    x, y = lm_batch(rows); x, y = x.to(acc.device), y.to(acc.device)
    with acc.autocast():
        _, loss = model(x, y)
    acc.backward(loss)
    if acc.sync_gradients:
        opt.step(); opt.zero_grad()
    losses.append(loss.item())
print(f"device={acc.device}  loss first 5 avg {sum(losses[:5])/5:.3f} -> last 5 avg {sum(losses[-5:])/5:.3f}")

device=cpu  loss first 5 avg 3.454 -> last 5 avg 1.531


### 9. Real Hugging Face libraries (optional, guarded)

**What this cell does (plain language):** if `transformers` / `tokenizers` / `datasets` / `accelerate` are installed, we use their real APIs with only locally constructed objects: a randomly initialised `GPT2Config` model (no download), a `tokenizers` BPE trained on our corpus, a `datasets.Dataset.from_dict`, and a real `Accelerator`. If a library is missing we just print the install command, so the notebook still runs everywhere.

In [9]:
# --- transformers: random-init model from a local config (no network) ---
try:
    from transformers import GPT2Config, GPT2LMHeadModel
    hf_cfg = GPT2Config(vocab_size=len(bpe), n_positions=32, n_embd=32, n_layer=2, n_head=2)
    hf_model = GPT2LMHeadModel(hf_cfg)
    out = hf_model(input_ids=x, labels=x)
    print("transformers GPT2 params:", sum(p.numel() for p in hf_model.parameters()), "| loss:", out.loss.item())
except ImportError:
    print("transformers not installed -> pip install transformers")

# --- tokenizers: train a real BPE on our in-memory corpus ---
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    t = Tokenizer(models.BPE(unk_token="<unk>")); t.pre_tokenizer = pre_tokenizers.Whitespace()
    t.train_from_iterator(corpus, trainers.BpeTrainer(vocab_size=60, special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]))
    print("tokenizers:", t.encode("the lowest cat").tokens)
except ImportError:
    print("tokenizers not installed -> pip install tokenizers")

# --- datasets: Dataset.from_dict + map ---
try:
    from datasets import Dataset
    hd = Dataset.from_dict({"text": corpus}).map(lambda r: {"n_chars": len(r["text"])})
    print("datasets:", hd[0])
except ImportError:
    print("datasets not installed -> pip install datasets")

# --- accelerate: the real Accelerator ---
try:
    from accelerate import Accelerator
    a = Accelerator(cpu=True, gradient_accumulation_steps=2)
    print("accelerate device:", a.device, "| mixed precision:", a.mixed_precision)
except ImportError:
    print("accelerate not installed -> pip install accelerate")

transformers not installed -> pip install transformers
tokenizers not installed -> pip install tokenizers
datasets not installed -> pip install datasets
accelerate not installed -> pip install accelerate


### 10. Mapping our pieces onto the real HF API

**What this cell does (plain language):** no new computation: we print a lookup table linking each thing we built to its Hugging Face equivalent, and confirm numerically that the loss on a `labels=input_ids` batch uses the shift-by-one convention (what `GPT2LMHeadModel(labels=...)` does internally).

In [10]:
mapping = {
    "CharTokenizer / BPETokenizer": "tokenizers.Tokenizer / AutoTokenizer",
    "collate (pad + mask + -100)":   "DataCollatorForLanguageModeling / DataCollatorWithPadding",
    "MiniDataset.map/filter/split":  "datasets.Dataset.map / filter / train_test_split",
    "GPTConfig + save/from_pretrained": "PretrainedConfig + PreTrainedModel.save_pretrained / from_pretrained",
    "generate()":                    "model.generate(do_sample, temperature, top_k)",
    "MiniAccelerator":               "accelerate.Accelerator (prepare, accumulate, backward)",
}
for k, v in mapping.items(): print(f"{k:35s} -> {v}")

# Shift-by-one check: manual loss equals the model's loss when we pass shifted targets.
model.eval(); ids = torch.tensor([bpe.encode("the lowest cat")])
logits, _ = model(ids)
manual = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1))
_, builtin = model(ids[:, :-1], ids[:, 1:])
print("manual shifted loss:", manual.item(), "| model loss with shifted targets:", builtin.item())

CharTokenizer / BPETokenizer        -> tokenizers.Tokenizer / AutoTokenizer
collate (pad + mask + -100)         -> DataCollatorForLanguageModeling / DataCollatorWithPadding
MiniDataset.map/filter/split        -> datasets.Dataset.map / filter / train_test_split
GPTConfig + save/from_pretrained    -> PretrainedConfig + PreTrainedModel.save_pretrained / from_pretrained
generate()                          -> model.generate(do_sample, temperature, top_k)
MiniAccelerator                     -> accelerate.Accelerator (prepare, accumulate, backward)
manual shifted loss: 3.054367780685425 | model loss with shifted targets: 3.054367780685425


## Common bugs

- **Forgetting the attention mask / pad id.** Padded batches without a mask let real tokens attend to padding. Fix: always pass `attention_mask` (or use a causal model where right-padding is harmless but labels are still `-100`).
- **Padding tokens counted in the loss.** Labels equal to `pad_id` instead of `-100` teach the model to predict padding. Fix: set labels to `-100` at pad positions (and for any tokens you do not want to train on).
- **Left vs right padding for generation.** Decoder-only models must be left-padded for batched `generate`, otherwise new tokens are appended after pad tokens. Fix: `tokenizer.padding_side = 'left'` at inference.
- **Tokenizer / model vocab mismatch.** Adding special tokens grows the tokenizer; if you do not `resize_token_embeddings`, ids go out of range (index errors or silent garbage). Fix: always resize the embedding after changing the vocab.
- **Not shifting labels.** Computing `CE(logits, input_ids)` trains the model to copy the current token. Fix: predict `ids[:, 1:]` from `logits[:, :-1]` (HF does this for you inside the model).
- **Gradient accumulation without loss scaling.** Summing N un-scaled micro-batch losses makes the effective LR N times bigger. Fix: divide the loss by N (what `accelerator.accumulate` handles).
- **`.map` returning nothing / mutating in place.** `Dataset.map` must return a dict of new columns; forgetting `return` silently leaves the dataset unchanged. Fix: return the dict, and inspect `column_names`.
- **Saving only the weights.** A `state_dict` without `config.json` (and tokenizer files) cannot be reloaded by someone else. Fix: save config, weights and tokenizer together.

## Exercises

**Exercise 1.** Extend `CharTokenizer` with a `pad_batch(list_of_texts)` method that returns padded ids and an attention mask.

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
def pad_batch(tok, texts):
    ids = [tok.encode(t) for t in texts]
    b = collate(ids, pad_id=tok.pad_id)
    return b["input_ids"], b["attention_mask"]
ii, mm = pad_batch(tok, ["the cat", "a dog"])
print(ii); print(mm)

tensor([[ 2, 18, 10,  8,  4,  6,  5, 18,  3],
        [ 2,  5,  4,  7, 15,  9,  3,  0,  0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0]])


**Exercise 2.** Train BPE with 5 merges and with 40 merges on the corpus. Compare the token count of `'lowest newest'` and explain the trade-off.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
for n in (5, 40):
    m = train_bpe(corpus * 3, n)
    toks = [s for w in "lowest newest".split() for s in bpe_encode_word(w, m)]
    print(n, "merges ->", len(toks), "tokens:", toks)
# More merges => fewer, longer tokens (shorter sequences) but a bigger embedding table / vocab.

5 merges -> 10 tokens: ['l', 'o', 'we', 's', 't</w>', 'n', 'e', 'we', 's', 't</w>']
40 merges -> 2 tokens: ['lowest</w>', 'newest</w>']


**Exercise 3.** Use `MiniDataset.map(batched=True)` to add a `length` column computed for all rows at once.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

In [16]:
# Solution 3
ds2 = ds.map(lambda cols: {"length": [len(x) for x in cols["input_ids"]]}, batched=True)
print(ds2.column_names, ds2["length"][:5])

['text', 'label', 'input_ids', 'n_tokens', 'length'] [10, 11, 16, 9, 13]


**Exercise 4.** Implement top-p (nucleus) filtering for the logits of a single step and test that the kept probability mass is >= p.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
def top_p_filter(logits, p=0.9):
    probs = logits.softmax(-1)
    sp, si = probs.sort(descending=True)
    cum = sp.cumsum(-1)
    remove = cum - sp > p            # drop tokens whose preceding mass already exceeds p
    sp[remove] = 0
    out = torch.full_like(probs, 0.0); out.scatter_(-1, si, sp)
    return out / out.sum(-1, keepdim=True), (probs * (out > 0)).sum(-1)
lg = torch.randn(1, 50)
newp, kept = top_p_filter(lg, 0.9)
print("kept mass of original distribution:", kept.item(), ">= 0.9:", kept.item() >= 0.9)

kept mass of original distribution: 0.9034181833267212 >= 0.9: True


**Exercise 5.** Show that gradient accumulation with `grad_accum=4` and micro-batch 2 gives the same gradient as one batch of 8 (use a model without dropout).

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
torch.manual_seed(0)
m = GPT(cfg); xs = torch.randint(4, len(bpe), (8, 12)); ys = torch.randint(4, len(bpe), (8, 12))
m.zero_grad(); m(xs, ys)[1].backward(); g_full = m.head.weight.grad.clone()
m.zero_grad()
for i in range(4):
    (m(xs[2*i:2*i+2], ys[2*i:2*i+2])[1] / 4).backward()
print("max abs diff:", (g_full - m.head.weight.grad).abs().max().item())

max abs diff: 1.6763806343078613e-08


**Exercise 6.** Save the BPE tokenizer and model into one directory and write a `load_all(dir)` that returns both.

In [21]:
# Your attempt for Exercise 6 here

**Solution 6**

In [22]:
# Solution 6
def load_all(d):
    return BPETokenizer.load(os.path.join(d, "tokenizer.json")), from_pretrained(d)
with tempfile.TemporaryDirectory() as d:
    save_pretrained(model, d); bpe.save(os.path.join(d, "tokenizer.json"))
    t2, m2 = load_all(d)
print(len(t2), sum(p.numel() for p in m2.parameters()))

35 27616
